In [8]:
# !pip install torch torchvision scikit-learn matplotlib pandas

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split, ConcatDataset
from torchvision import datasets, transforms, models
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [2]:
def get_dataloaders(dataset_name='MNIST', batch_size=16):
    # ResNet expects 3 channels usually, but we will modify the model.
    # However, ResNet downsamples significantly. We resize to 32x32 or 64x64 to avoid issues.
    # 224x224 is standard for ResNet but very slow for training. Using 32x32 for speed.
    transform = transforms.Compose([
        transforms.Resize((32, 32)),
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])

    if dataset_name == 'MNIST':
        train_full = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
        test_full = datasets.MNIST(root='./data', train=False, download=True, transform=transform)
    elif dataset_name == 'FashionMNIST':
        train_full = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
        test_full = datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

    # 1. Merge Datasets
    full_dataset = ConcatDataset([train_full, test_full])
    total_len = len(full_dataset)

    # 2. Calculate Split Sizes (70% - 10% - 20%)
    train_size = int(0.7 * total_len)
    val_size = int(0.1 * total_len)
    test_size = total_len - train_size - val_size

    # 3. Perform Split
    train_ds, val_ds, test_ds = random_split(full_dataset, [train_size, val_size, test_size])

    # 4. Create Loaders
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, pin_memory=True)

    print(f"{dataset_name} Split -> Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}")
    return train_loader, val_loader, test_loader

In [3]:
def get_resnet_model(model_name='resnet18', pretrained=False):
    if model_name == 'resnet18':
        model = models.resnet18(weights=None) # pretrained=False
    elif model_name == 'resnet50':
        model = models.resnet50(weights=None) # pretrained=False

    # MODIFY FIRST LAYER: Change in_channels from 3 (RGB) to 1 (Grayscale)
    # Original: nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False)
    model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)

    return model.to(device)

In [4]:
def train_model(model, train_loader, val_loader, optimizer, epochs=5):
    criterion = nn.CrossEntropyLoss()
    scaler = torch.amp.GradScaler('cuda') # For AMP

    train_losses, val_accuracies = [], []

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()

            # AMP Forward Pass
            with torch.amp.autocast('cuda'):
                outputs = model(images)
                loss = criterion(outputs, labels)

            # AMP Backward Pass
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item()

        # Validation
        val_acc = evaluate_model(model, val_loader)
        train_losses.append(running_loss / len(train_loader))
        val_accuracies.append(val_acc)

        print(f"Epoch {epoch+1}/{epochs} - Loss: {running_loss/len(train_loader):.4f} - Val Acc: {val_acc:.2f}%")

    return train_losses, val_accuracies

def evaluate_model(model, loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return 100 * correct / total

In [ ]:
# Experiment Configuration
datasets_list = ['MNIST', 'FashionMNIST']
# You can uncomment all configs. Using fewer here for testing speed.
configs = [
    {'batch': 16, 'optim': 'SGD',  'lr': 0.001, 'model': 'resnet18'},
    {'batch': 16, 'optim': 'SGD',  'lr': 0.0001, 'model': 'resnet18'},
    {'batch': 16, 'optim': 'Adam', 'lr': 0.001, 'model': 'resnet18'},
    {'batch': 16, 'optim': 'Adam', 'lr': 0.0001, 'model': 'resnet18'},
    # Add ResNet-50 and other configs here...
]

results = []

print("Starting Deep Learning Experiments...")

for ds_name in datasets_list:
    print(f"\n--- Processing {ds_name} ---")

    for conf in configs:
        print(f"Running: {conf}")

        # 1. Setup Data
        train_loader, val_loader, test_loader = get_dataloaders(ds_name, conf['batch'])

        # 2. Setup Model
        model = get_resnet_model(conf['model'])

        # 3. Setup Optimizer
        if conf['optim'] == 'SGD':
            optimizer = optim.SGD(model.parameters(), lr=conf['lr'], momentum=0.9)
        else:
            optimizer = optim.Adam(model.parameters(), lr=conf['lr'])

        # 4. Train (Using 2 epochs for demo, INCREASE TO 5-10 for actual assignment)
        train_losses, val_accs = train_model(model, train_loader, val_loader, optimizer, epochs=3)

        # 5. Final Test
        test_acc = evaluate_model(model, test_loader)

        # 6. Store Result
        results.append({
            'Dataset': ds_name,
            'Model': conf['model'],
            'Batch': conf['batch'],
            'Optim': conf['optim'],
            'LR': conf['lr'],
            'Test Acc': test_acc
        })
        print(f"Result: {test_acc:.2f}%")

# Create DataFrame for easy viewing
df_results = pd.DataFrame(results)
print("\nFinal Results Table:")
print(df_results)

Starting Deep Learning Experiments...

--- Processing MNIST ---
Running: {'batch': 16, 'optim': 'SGD', 'lr': 0.001, 'model': 'resnet18'}


100.0%
100.0%
100.0%
100.0%


MNIST Split -> Train: 49000, Val: 7000, Test: 14000
Epoch 1/3 - Loss: 0.1898 - Val Acc: 98.60%
Epoch 2/3 - Loss: 0.0536 - Val Acc: 98.73%
Epoch 3/3 - Loss: 0.0330 - Val Acc: 99.01%
Result: 98.97%
Running: {'batch': 16, 'optim': 'SGD', 'lr': 0.0001, 'model': 'resnet18'}
MNIST Split -> Train: 49000, Val: 7000, Test: 14000
Epoch 1/3 - Loss: 0.4419 - Val Acc: 96.73%
Epoch 2/3 - Loss: 0.1041 - Val Acc: 97.69%
Epoch 3/3 - Loss: 0.0717 - Val Acc: 98.19%
Result: 98.19%
Running: {'batch': 16, 'optim': 'Adam', 'lr': 0.001, 'model': 'resnet18'}
MNIST Split -> Train: 49000, Val: 7000, Test: 14000
Epoch 1/3 - Loss: 0.1924 - Val Acc: 97.57%
Epoch 2/3 - Loss: 0.0888 - Val Acc: 98.24%
Epoch 3/3 - Loss: 0.0679 - Val Acc: 98.50%
Result: 98.46%
Running: {'batch': 16, 'optim': 'Adam', 'lr': 0.0001, 'model': 'resnet18'}
MNIST Split -> Train: 49000, Val: 7000, Test: 14000
Epoch 1/3 - Loss: 0.2340 - Val Acc: 98.11%
Epoch 2/3 - Loss: 0.0752 - Val Acc: 98.86%
Epoch 3/3 - Loss: 0.0513 - Val Acc: 98.34%
Result: 

100.0%
100.0%
100.0%
100.0%


FashionMNIST Split -> Train: 49000, Val: 7000, Test: 14000
Epoch 1/3 - Loss: 0.5381 - Val Acc: 87.34%
Epoch 2/3 - Loss: 0.3391 - Val Acc: 88.41%
Epoch 3/3 - Loss: 0.2792 - Val Acc: 89.90%
Result: 89.96%
Running: {'batch': 16, 'optim': 'SGD', 'lr': 0.0001, 'model': 'resnet18'}
FashionMNIST Split -> Train: 49000, Val: 7000, Test: 14000
Epoch 1/3 - Loss: 0.7221 - Val Acc: 86.47%
Epoch 2/3 - Loss: 0.3984 - Val Acc: 87.79%
Epoch 3/3 - Loss: 0.3401 - Val Acc: 88.70%
Result: 88.55%
Running: {'batch': 16, 'optim': 'Adam', 'lr': 0.001, 'model': 'resnet18'}
FashionMNIST Split -> Train: 49000, Val: 7000, Test: 14000
Epoch 1/3 - Loss: 0.5242 - Val Acc: 86.83%
Epoch 2/3 - Loss: 0.3643 - Val Acc: 88.66%
Epoch 3/3 - Loss: 0.3145 - Val Acc: 90.53%


In [ ]:
def run_svm_experiment(dataset_name):
    print(f"\nRunning SVM for {dataset_name}...")

    # Load Data (Standard MNIST loaders are fine for SVM, we just need vectors)
    if dataset_name == 'MNIST':
        train_data = datasets.MNIST(root='./data', train=True, download=True, transform=transforms.ToTensor())
        test_data = datasets.MNIST(root='./data', train=False, download=True, transform=transforms.ToTensor())
    else:
        train_data = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transforms.ToTensor())
        test_data = datasets.FashionMNIST(root='./data', train=False, download=True, transform=transforms.ToTensor())

    # Flatten Data: (N, 28, 28) -> (N, 784)
    # WARNING: Training SVM on 60k images takes HOURS. Subsampling to 10k for demonstration.
    # REMOVE [:10000] FOR FULL ASSIGNMENT SUBMISSION if required.
    X_train = train_data.data.numpy().reshape(-1, 28*28)[:10000]
    y_train = train_data.targets.numpy()[:10000]

    X_test = test_data.data.numpy().reshape(-1, 28*28)
    y_test = test_data.targets.numpy()

    kernels = ['poly', 'rbf']

    for k in kernels:
        clf = SVC(kernel=k)

        start_time = time.time()
        clf.fit(X_train, y_train)
        end_time = time.time()

        train_time_ms = (end_time - start_time) * 1000

        pred = clf.predict(X_test)
        acc = accuracy_score(y_test, pred) * 100

        print(f"Kernel: {k} | Train Time: {train_time_ms:.2f} ms | Accuracy: {acc:.2f}%")

# Run SVM
run_svm_experiment('MNIST')
run_svm_experiment('FashionMNIST')


Running SVM for MNIST...
Kernel: poly | Train Time: 8531.36 ms | Accuracy: 95.15%
Kernel: rbf | Train Time: 7839.94 ms | Accuracy: 95.94%

Running SVM for FashionMNIST...
Kernel: poly | Train Time: 8651.86 ms | Accuracy: 81.66%
Kernel: rbf | Train Time: 8416.98 ms | Accuracy: 85.31%


In [ ]:
#Q2
!pip install thop
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
import time
from thop import profile # For FLOPs counting

In [ ]:
# --- CONFIGURATION ---
BATCH_SIZE = 16
LR = 0.001
EPOCHS_CPU = 1  # Keep low (1) for CPU to save time
EPOCHS_GPU = 3  # Can be higher for GPU
DATASET_NAME = 'FashionMNIST'

# --- 1. DATA SETUP ---
# ResNet requires 3 channels, we will modify model later.
# We resize to 32x32 for speed (Standard ResNet is 224x224, but that is too heavy for CPU testing)
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# Download Data
train_data = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
test_data = datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

# --- 2. MODEL BUILDER ---
def get_model(model_name, device):
    if model_name == 'resnet18':
        model = models.resnet18(weights=None)
    elif model_name == 'resnet50':
        model = models.resnet50(weights=None)

    # Modify for 1 Channel (Grayscale)
    model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)

    return model.to(device)

100%|██████████| 26.4M/26.4M [00:01<00:00, 17.1MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 250kB/s]
100%|██████████| 4.42M/4.42M [00:00<00:00, 5.09MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 15.5MB/s]


In [ ]:
def calculate_flops(model, device):
    # Create a dummy input of size (Batch_Size, Channel, Height, Width)
    dummy_input = torch.randn(1, 1, 32, 32).to(device)
    macs, params = profile(model, inputs=(dummy_input, ), verbose=False)

    # FLOPs is roughly 2 * MACs (Multiply-Accumulates)
    flops = 2 * macs
    return flops

In [9]:
def run_experiment(device_type, model_name, optimizer_name):
    # Setup Device
    device = torch.device("cuda" if device_type == "GPU" and torch.cuda.is_available() else "cpu")
    print(f"   -> Running on {device}...")

    # Load Model
    model = get_model(model_name, device)

    # Calculate FLOPs (Only need to do this once per model type)
    flops = calculate_flops(model, device)

    # Loaders
    train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
    test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False)

    # Optimizer
    if optimizer_name == 'SGD':
        optimizer = optim.SGD(model.parameters(), lr=LR)
    else:
        optimizer = optim.Adam(model.parameters(), lr=LR)

    criterion = nn.CrossEntropyLoss()

    # Determine Epochs
    epochs = EPOCHS_CPU if device_type == 'CPU' else EPOCHS_GPU

    # --- START TIMER ---
    start_time = time.time()

    model.train()
    for epoch in range(epochs):
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

    # --- END TIMER ---
    end_time = time.time()
    total_train_time_ms = (end_time - start_time) * 1000  # Convert to ms

    # Evaluation
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total

    return accuracy, total_train_time_ms, flops



In [10]:
experiments = [
    # CPU Experiments
    {'compute': 'CPU', 'model': 'resnet18', 'optim': 'SGD'},
    {'compute': 'CPU', 'model': 'resnet18', 'optim': 'Adam'},
    {'compute': 'CPU', 'model': 'resnet50', 'optim': 'SGD'}, # Heavy!

    # GPU Experiments
    {'compute': 'GPU', 'model': 'resnet18', 'optim': 'SGD'},
    {'compute': 'GPU', 'model': 'resnet18', 'optim': 'Adam'},
    {'compute': 'GPU', 'model': 'resnet50', 'optim': 'SGD'},
    {'compute': 'GPU', 'model': 'resnet50', 'optim': 'Adam'},
]

print(f"{'Compute':<8} | {'Model':<10} | {'Optim':<6} | {'Acc (%)':<8} | {'Time (ms)':<12} | {'FLOPs':<15}")
print("-" * 75)

for exp in experiments:
    acc, time_ms, flops = run_experiment(exp['compute'], exp['model'], exp['optim'])

    print(f"{exp['compute']:<8} | {exp['model']:<10} | {exp['optim']:<6} | {acc:<8.2f} | {time_ms:<12.0f} | {flops:<15.0f}")


Compute  | Model      | Optim  | Acc (%)  | Time (ms)    | FLOPs          
---------------------------------------------------------------------------
   -> Running on cpu...


KeyboardInterrupt: 